In [1]:
#cell 1: đọc manifest_all.csv, lấy tên clip và tính khối thời gian cho Hand, Bonus
import pandas as pd, re
from pathlib import Path
BASE = Path(r"C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321")
GAP = 300  # giây, giống notebook 27: nghỉ lâu hơn thì tính là khối mới
m = pd.read_csv(BASE / "manifest_all.csv")
m["fn"] = m.path.str.replace("\\", "/", regex=False).str.split("/").str[-1].str.replace(".npz", "", regex=False)
m["t"] = pd.to_datetime(m.fn.where(m.nguon != "dataset_labeled").str[4:18], format="%Y%m%d%H%M%S", errors="coerce")
m["khoi"] = 0
for ng in ["Dataset_hand", "Dataset_bonus"]:
    s = m[m.nguon == ng].sort_values("t")
    m.loc[s.index, "khoi"] = (s.t.diff().dt.total_seconds().fillna(1e9) > GAP).cumsum().values
print(len(m), "video; khối Hand:", m[m.nguon == "Dataset_hand"].khoi.max(), "; khối Bonus:", m[m.nguon == "Dataset_bonus"].khoi.max())

305 video; khối Hand: 7 ; khối Bonus: 6


In [2]:
#cell 2: gán nhóm phiên quay và tập mới theo quy tắc đã thống nhất
LAB = {"IMG_0510": "L1", "IMG_0510(1)": "L2", "IMG_1383": "L3", "IMG_8171": "L4", "IMG_8172": "L5", "IMG_8173": "L5"}
HAND = {1: "H1", 2: "H1", 3: "H1", 4: "H2", 5: "H2", 6: "H3", 7: "H3"}   # khối thời gian -> phiên quay
BONUS = {1: "B1", 2: "B2", 3: "B2", 4: "B2", 5: "B2", 6: "B3"}
TAP = {"L3": "test", "B1": "test", "B3": "test", "L5": "val", "H2": "val"}  # nhóm còn lại: train
def gan(r):
    if r.nguon == "dataset_labeled": return LAB[re.sub(r"_\d+$", "", r.fn)]
    return (HAND if r.nguon == "Dataset_hand" else BONUS)[r.khoi]
m["nhom"] = m.apply(gan, axis=1)
m["tap_moi"] = m.nhom.map(TAP).fillna("train")

In [3]:
#cell 3: kiểm tra nhóm không giao nhau giữa các tập, số clip khớp bảng, file landmark còn đủ
EXPECT = {"L1": 36, "L2": 50, "L3": 15, "L4": 6, "L5": 27, "H1": 33, "H2": 18, "H3": 30, "B1": 12, "B2": 66, "B3": 12}
sz = m.groupby("nhom").size().to_dict()
assert sz == EXPECT, f"Số clip từng nhóm lệch bảng: {sz}"
assert (m.groupby("nhom").tap_moi.nunique() == 1).all(), "Có nhóm nằm ở nhiều tập"
print("OK: 11 nhóm đúng số clip, không nhóm nào nằm ở 2 tập\n")
print(pd.crosstab(m.nhom, m.tap_moi), "\n")
print(pd.crosstab(m.tap_moi, m.lop, margins=True), "\n")
print(pd.crosstab(m.tap_moi, m.nguon, margins=True), "\n")
print("Tập cũ (hàng) so với tập mới (cột):"); print(pd.crosstab(m.tap, m.tap_moi, margins=True))
thieu = [p for p in m.path if not (BASE / p).exists()]
print("\nFile landmark thiếu:", len(thieu), thieu[:3])

OK: 11 nhóm đúng số clip, không nhóm nào nằm ở 2 tập

tap_moi  test  train  val
nhom                     
B1         12      0    0
B2          0     66    0
B3         12      0    0
H1          0     33    0
H2          0      0   18
H3          0     30    0
L1          0     36    0
L2          0     50    0
L3         15      0    0
L4          0      6    0
L5          0      0   27 

lop      paper  rock  scissors  All
tap_moi                            
test        13    13        13   39
train       77    71        73  221
val         15    16        14   45
All        105   100       100  305 

nguon    Dataset_bonus  Dataset_hand  dataset_labeled  All
tap_moi                                                   
test                24             0               15   39
train               66            63               92  221
val                  0            18               27   45
All                 90            81              134  305 

Tập cũ (hàng) so với tập mới (cộ

In [4]:
#cell 4: lưu manifest_v2.csv (cột tap là tập mới, tap_cu là tập cũ, nhom là phiên quay); không đụng manifest_all.csv
out = m.drop(columns=["fn", "t", "khoi", "tap_moi"]).copy()
out["tap_cu"] = m.tap; out["tap"] = m.tap_moi; out["nhom"] = m.nhom
f = BASE / "manifest_v2.csv"
print("Đã có manifest_v2.csv, sẽ ghi đè" if f.exists() else "Tạo mới manifest_v2.csv")
out.to_csv(f, index=False)
print(out.columns.tolist(), len(out), "dòng")

Tạo mới manifest_v2.csv
['path', 'nguon', 'lop', 'nhan', 'khung', 'fps', 'tap', 'nhom', 'tap_cu'] 305 dòng
